In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import librosa

from song_recommender.paths import *

# Data Validation

We validate the following:

1. Dataframe length
2. No duplicate tracks in dataframe based on `spotify_id`
3. Number of audio files
4. Number of stems
5. Number of spectrogram pngs
6. Number of "raw" spectrograms saved as .npy
7. Image sizes are all identical (width = 862, height = 256)
8. Array shapes are all identical
8. Sound rates of original audio are identical (22050)

In [ ]:
# load dataframe and create Spec object
df = pd.read_parquet(DATA_DIR / 'metadata_raw.parquet')

1. Check correct length of dataframe: 11239

In [6]:
len(df)

11239

2. Check for any duplicate tracks based on identifier `spotify_id`.

In [7]:
df['spotify_id'].duplicated().sum()

np.int64(0)

In [8]:
df[df.duplicated('spotify_id', keep=False)]

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,crude_genre,duration_min,clip_duration


3. Check for correct number of audio files.

In [9]:
assert len(df) == len(list(AUDIO_DIR.rglob('*.wav')))
assert len(df) == len([f for f in AUDIO_DIR.rglob('*') if f.is_file() and f.name != '.DS_Store'])

4. Check for correct number of stems.

In [10]:
# correct number of directories
assert len(df) == len(list(STEMS_DIR.rglob('*/')))

# correct number of files
assert len(df)*4 == len(list(STEMS_DIR.rglob('*.wav')))
assert len(df)*4 == len([f for f in STEMS_DIR.rglob('*') if f.is_file() and f.name != '.DS_Store'])

5. Check for the correct number of spectrogram pngs.

In [11]:
# correct number of directories
assert len(df) == len(list(SPECTROGRAM_PNG_DIR.rglob('*/')))

# correct number of files
assert len(df)*5 == len(list(SPECTROGRAM_PNG_DIR.rglob('*.png')))
assert len(df)*5 == len([f for f in SPECTROGRAM_PNG_DIR.rglob('*') if f.is_file() 
                         and f.name != '.DS_Store'])

6. Check for the correct number of raw spectrograms

In [12]:
# correct number of directories
assert len(df) == len(list(SPECTROGRAM_RAW_DIR.rglob('*/')))

# correct number of files
assert len(df)*5 == len(list(SPECTROGRAM_RAW_DIR.rglob('*.npy')))
assert len(df)*5 == len([f for f in SPECTROGRAM_RAW_DIR.rglob('*') if f.is_file() 
                         and f.name != '.DS_Store'])

7. Check png dimensions.

In [13]:
images = Path(SPECTROGRAM_PNG_DIR).rglob('*.png') 

width = 862
height = 256

for image_path in images:
   if (width, height) != Image.open(image_path).size:
        print(f'{image_path.name} has dimensions {Image.open(image_path).size}')

8. Check raw spectrogram array shapes.

In [14]:
arrays = Path(SPECTROGRAM_RAW_DIR).rglob('*.npy') 

shape = (256,862)

for array_path in arrays:
    array = np.load(array_path)
    if shape != array.shape:
        print(f'{array_path.name} has shape {array.shape}')

9. Check sound rates.

In [15]:
audio = Path(AUDIO_DIR).rglob('*.wav')

base_rate = 22050

for audio_path in audio:
    y, sr = librosa.load(audio_path, 
                         duration = 10.0) # load only 1st 10 seconds
    if sr != base_rate:
        print(f'{audio_path.name} has sound rate {sr}')
    

In [16]:
stems = Path(STEMS_DIR).rglob('*.wav')

base_rate = 22050

for stem_path in stems:
    y, sr = librosa.load(stem_path, 
                         duration = 10.0) # load only 1st 10 seconds
    if sr != base_rate:
        print(f'{stem_path.name} has sound rate {sr}')